# Optimize the 3D Grating Coupler

Now that we know the 3D design and gradients look good, we can optimize the structure. Here we will preform non-stochastic optimization using the parameters from the 2D non-stochastic optimization as the starting parameters. 

In [1]:
import autograd.numpy as np 
import matplotlib.pyplot as plt

import tidy3d as td
import tidy3d.web as web
td.config.logging_level = "ERROR"

import pickle

from main import (make_sim, get_coupling_efficiency, projection_builder, run_adam,
                    R, r0_extra, initial_fill_factor, grating_period, 
                    etch_depth, to_substrate, N_teeth, n_wl, wl_range,
                    to_substrate)


14:12:53 EDT WARNING: Using canonical configuration directory at                
             '/Users/dominic/.config/tidy3d'. Found legacy directory at         
             '~/.tidy3d', which will be ignored. Remove it manually or run      
             'tidy3d config migrate --delete-legacy' to clean up.               

In [ ]:
# define the load function to get the initial parameters
def load_history(filename):
    with open(filename, "rb") as f:
        file = pickle.load(f)
    return file

# define the objective function
def objective(params,projection=lambda x: x):

    # project the parameters
    params_proj = projection(params)

    # unpack the projected parameters
    widths = params_proj[:-3]
    r0 = params_proj[-3]
    etch_depth = params_proj[-2]
    to_substrate = params_proj[-1]
    
    # run the simulation
    sim = make_sim(widths, r0=r0, etch_depth=etch_depth, to_substrate=to_substrate, include_field_monitor=False)
    sim_data = web.run(sim, task_name="GC4um_3D_initial_opt", verbose=False, path="data/tidy3d_output/tmp.hdf5")
    coupling = get_coupling_efficiency(sim_data)
    return np.mean(coupling)

Now we can load the previous result and run the optimization. 

In [7]:
# load the previous parameters
data = load_history("../GC_4um_2D/data/opt/history.pkl")
params = data['history']['params'][-1]

# make the grating structure
widths = params[:-3]
r0 = params[-3] + r0_extra # need to add the taper length
etch_depth = params[-2]
to_substrate = params[-3]

# re-pack the parameters in an autograd array 
params = np.concatenate([widths, [r0], [etch_depth], [to_substrate]])

# define the projection for enforcing constraints
project, inverse_project = projection_builder()

# run the optimization
history, opt_state = run_adam(params, project, inverse_project, objective, num_steps=50, learning_rate=0.01, verbose=True)

step = 1
	J = 4.435e-01
	grad_norm = 6.8530e-01
step = 2
	J = 4.603e-01
	grad_norm = 6.5406e-01
	angle = 9.177584727803579
step = 3
	J = 4.739e-01
	grad_norm = 6.4442e-01
	angle = 10.421820544788465
step = 4
	J = 4.857e-01
	grad_norm = 6.3573e-01
	angle = 7.352723033119186
step = 5
	J = 4.968e-01
	grad_norm = 6.3511e-01
	angle = 4.862150738588306
step = 6
	J = 5.082e-01
	grad_norm = 6.1964e-01
	angle = 1.7880730949959451
step = 7
	J = 5.193e-01
	grad_norm = 5.9641e-01
	angle = 0.536242742615617
step = 8
	J = 5.305e-01
	grad_norm = 5.6911e-01
	angle = 1.2202016433235907
step = 9
	J = 5.416e-01
	grad_norm = 5.3900e-01
	angle = 2.469585570710766
step = 10
	J = 5.522e-01
	grad_norm = 5.0992e-01
	angle = 3.0592785204950013
step = 11
	J = 5.621e-01
	grad_norm = 4.8382e-01
	angle = 3.618406938209354
step = 12
	J = 5.714e-01
	grad_norm = 4.5862e-01
	angle = 3.5990226023877296
step = 13
	J = 5.801e-01
	grad_norm = 4.3645e-01
	angle = 3.196510792079571
step = 14
	J = 5.890e-01
	grad_norm = 4.153

In [9]:
# save history and opt state in a pickle file
with open('data/3d_opt/history.pkl', 'wb') as f:
    pickle.dump({'history': history, 'opt_state': opt_state}, f)

# Analysis 

Now we analyze the results of the optimization

In [ ]:
# define the initial simulation
sim_initial = make_sim(widths, r0=r0, etch_depth=etch_depth, to_substrate=to_substrate, include_field_monitor=True)

# pull out the final parameters
widths_final = history["params"][-1][:-3]
r0_final = history["params"][-1][-3]
etch_depth_final = history["params"][-1][-2]
to_substrate_final = history["params"][-1][-1]

#define the final simulation 
sim_final = make_sim(widths_final, r0=r0_final, etch_depth=etch_depth_final, to_substrate=to_substrate_final, include_field_monitor=True)

#run the two simulations 
sim_data_initial = web.run(sim_initial, task_name="GC4um_3D_initial_opt", verbose=False, path="data/tidy3d_output/tmp.hdf5")
sim_data_final = web.run(sim_final, task_name="GC4um_3D_final_opt", verbose=False, path="data/tidy3d_output/tmp.hdf5")

#compute the coupling efficiencies 
ce_initial = get_coupling_efficiency(sim_data_initial)
ce_final = get_coupling_efficiency(sim_data_final)

In [ ]:
fig,ax = plt.subplots(2,2,dpi=300)

#plot the coupling efficiencies 
ax[0,0].plot(wl_range, ce_initial,label="Initial")
ax[0,0].plot(wl_range, ce_final,label="Final")
ax[0,0].set_xlabel("Wavelength (um)")
ax[0,0].set_ylabel("Coupling Efficiency")
ax[0,0].legend()

#plot the field monitors 
sim_data_initial.plot_field("field", "E", "abs^2", y=0, ax=ax[0,1],eps_alpha=0.5)
ax[0,1].set_title("Initial")
sim_data_final.plot_field("field", "E", "abs^2", y=0, ax=ax[1,1],eps_alpha=0.5)
ax[1,1].set_title("Final")

# plot the change in the objective function 
ax[1,0].plot(history["J"])
ax[1,0].set_xlabel("Iteration")
ax[1,0].set_ylabel("Objective Function")

plt.tight_layout()
plt.show()